In [ ]:
# ============================================================
# GRIDS ICU-MM — Complete Pipeline (Run from top every session)
# ============================================================

# ── Cell 1: Imports and Drive Mount ──────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import timedelta
import time
import threading
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted ✓


In [ ]:
# ── Cell 2: Keep-Alive Thread ─────────────────────────────────
def keep_alive():
    while True:
        time.sleep(60)
        print(".", end="", flush=True)

In [ ]:
thread = threading.Thread(target=keep_alive, daemon=True)
thread.start()
print("Keep-alive started ✓")

Keep-alive started ✓


In [ ]:
# ── Cell 3: Define Paths ──────────────────────────────────────
data_dir = Path("/content/drive/MyDrive/USC/ICU-MM-main/ICU-MM/data/comb")
save_dir = Path("/content/drive/MyDrive/USC/ICU-MM-main/outputs")
save_dir.mkdir(parents=True, exist_ok=True)
print("Paths defined ✓")

Paths defined ✓


In [ ]:
# ── Cell 4: Load Raw Files ────────────────────────────────────
cohort = pd.read_csv(data_dir / "cohort.csv")

In [ ]:
labs_filtered = pd.read_csv(data_dir / "labs.csv", parse_dates=["lab_time"])

In [ ]:
prescriptions = pd.read_csv(data_dir / "prescriptions.csv",
                             parse_dates=["med_starttime", "med_stoptime"])

.

In [ ]:
resp_labels   = pd.read_csv(data_dir / "respiratory_failure_labels.csv")

In [ ]:
labs_filtered["hadm_id"] = labs_filtered["hadm_id"].astype("int64")

In [ ]:
print(f"cohort:        {cohort.shape}")
print(f"labs_filtered: {labs_filtered.shape}")
print(f"prescriptions: {prescriptions.shape}")
print(f"resp_labels:   {resp_labels.shape}")
print("Raw files loaded ✓")

cohort:        (94458, 7)
labs_filtered: (35532871, 6)
prescriptions: (8482752, 8)
resp_labels:   (94458, 4)
Raw files loaded ✓


In [ ]:
# ── Cell 5: Build cohort_clean ────────────────────────────────
cohort["icu_intime"]  = pd.to_datetime(cohort["icu_intime"])
cohort["icu_outtime"] = pd.to_datetime(cohort["icu_outtime"])

cohort["stay_duration_hours"] = (
    cohort["icu_outtime"] - cohort["icu_intime"]
).dt.total_seconds() / 3600

In [ ]:
# Remove bouncebacks (gap < 24h between consecutive stays)
stays_per_adm = cohort.groupby("hadm_id")["stay_id"].nunique()
multi_stay = cohort[cohort["hadm_id"].isin(stays_per_adm[stays_per_adm > 1].index)].copy()
multi_stay = multi_stay.sort_values(["hadm_id", "icu_intime"])
multi_stay["prev_outtime"] = multi_stay.groupby("hadm_id")["icu_outtime"].shift(1)
multi_stay["gap_hours"] = (multi_stay["icu_intime"] - multi_stay["prev_outtime"]).dt.total_seconds() / 3600
short_gap_stays = multi_stay[multi_stay["gap_hours"] < 24]["stay_id"].tolist()

In [ ]:
cohort_clean = cohort[~cohort["stay_id"].isin(short_gap_stays)].copy().reset_index(drop=True)

In [ ]:
# Administrative features
cohort_clean["obs_window_hours"] = (
    cohort_clean["stay_duration_hours"].clip(upper=12).fillna(12.0))
cohort_clean["is_short_stay"] = (cohort_clean["stay_duration_hours"]<12).astype(int).fillna(0)
cohort_clean["hadm_id"] = cohort_clean["hadm_id"].astype("int64")

In [ ]:
# Add respiratory failure label and rf_time
cohort_clean = cohort_clean.merge(resp_labels[["stay_id", "respiratory_failure", "rf_time"]],on="stay_id",how="left")
cohort_clean["respiratory_failure"] = (cohort_clean["respiratory_failure"].fillna(0).astype(int))
cohort_clean["rf_time"] = pd.to_datetime(cohort_clean["rf_time"], errors="coerce")

In [ ]:
# Feature cutoff — critical for leakage prevention
# Positive cases: cutoff = rf_time (stop before outcome)
# Negative cases: cutoff = icu_intime + 12h
cohort_clean["feature_cutoff"] = cohort_clean.apply(
    lambda row: (
        row["rf_time"]
        if row["respiratory_failure"] == 1 and pd.notna(row["rf_time"])
        else row["icu_intime"] + pd.Timedelta(hours=12)
    ),
    axis=1
)

cohort_clean["feature_window_hours"] = (
    cohort_clean["feature_cutoff"] - cohort_clean["icu_intime"]
).dt.total_seconds() / 3600

In [ ]:
# Save complete cohort_clean immediately
cohort_clean.to_csv(save_dir / "cohort_clean.csv", index=False)

In [ ]:
# Verify
print(f"cohort_clean shape:      {cohort_clean.shape}")
print(f"stay_id unique:          {cohort_clean['stay_id'].nunique() == len(cohort_clean)}")
print(f"Missing feature_cutoff:  {cohort_clean['feature_cutoff'].isna().sum()}")
print(f"Label distribution:")
print(cohort_clean["respiratory_failure"].value_counts())
print("cohort_clean built and saved ✓")


cohort_clean shape:      (91262, 14)
stay_id unique:          True
Missing feature_cutoff:  0
Label distribution:
respiratory_failure
0    56434
1    34828
Name: count, dtype: int64
cohort_clean built and saved ✓


In [ ]:
# ── Cell 6: Constants ─────────────────────────────────────────
TOP_LABS = [
    "Glucose", "Potassium", "Sodium", "Chloride",
    "Hemoglobin", "Creatinine", "Urea Nitrogen", "Bicarbonate",
    "Hematocrit", "Anion Gap", "Magnesium", "Platelet Count",
    "White Blood Cells", "MCHC", "Red Blood Cells", "MCV",
    "MCH", "RDW", "Phosphate", "Calcium, Total"
]
WINDOW_START_HOURS = -6
WINDOW_END_HOURS   = 12
print(f"TOP_LABS: {len(TOP_LABS)} labs")
print("Constants defined ✓")

TOP_LABS: 20 labs
Constants defined ✓


In [ ]:
# ── Cell 7: Extractor Functions ───────────────────────────────
def _set_nan(features, safe_name):
    """Set all features to NaN/0 when no lab data exists."""
    features[f"{safe_name}_mean"]       = np.nan
    features[f"{safe_name}_min"]        = np.nan
    features[f"{safe_name}_max"]        = np.nan
    features[f"{safe_name}_count"]      = 0
    features[f"{safe_name}_first"]      = np.nan
    features[f"{safe_name}_last"]       = np.nan
    features[f"{safe_name}_delta"]      = np.nan
    features[f"{safe_name}_early_mean"] = np.nan

In [ ]:
def extract_lab_features_v2(stay_row, stay_labs):
    """
    Extract lab features using feature_cutoff to prevent leakage.
    Positive cases: only use labs before rf_time
    Negative cases: use full 12h window
    8 features per lab × 20 labs + stay_id = 161 total
    """
    features   = {"stay_id": stay_row["stay_id"]}
    icu_intime = stay_row["icu_intime"]
    cutoff     = stay_row["feature_cutoff"]

    cutoff_hours = (
        cutoff - icu_intime
    ).total_seconds() / 3600

    if len(stay_labs) > 0:
        stay_labs = stay_labs.copy()
        stay_labs["hours"] = (
            stay_labs["lab_time"] - icu_intime
        ).dt.total_seconds() / 3600

        stay_labs = stay_labs[
            (stay_labs["hours"] >= WINDOW_START_HOURS) &
            (stay_labs["hours"] <= cutoff_hours)
        ].copy()

    for lab in TOP_LABS:
        safe_name = lab.replace(" ", "_").replace(",", "")

        if len(stay_labs) > 0 and lab in stay_labs["lab_name"].values:
            lab_rows          = stay_labs[stay_labs["lab_name"] == lab].copy()
            lab_rows["value"] = pd.to_numeric(
                lab_rows["value"], errors="coerce"
            )
            lab_rows          = lab_rows.dropna(subset=["value"])
            lab_rows          = lab_rows.sort_values(
                "hours").reset_index(drop=True)

            vals = lab_rows["value"]
            hrs  = lab_rows["hours"]

            if len(vals) == 0:
                _set_nan(features, safe_name)
            else:
                features[f"{safe_name}_mean"]  = vals.mean()
                features[f"{safe_name}_min"]   = vals.min()
                features[f"{safe_name}_max"]   = vals.max()
                features[f"{safe_name}_count"] = len(vals)
                features[f"{safe_name}_first"] = vals.iloc[0]
                features[f"{safe_name}_last"]  = vals.iloc[-1]
                features[f"{safe_name}_delta"] = vals.iloc[-1] - vals.iloc[0]

                early_mask = (hrs >= 0) & (hrs <= 6)
                features[f"{safe_name}_early_mean"] = (
                    vals[early_mask].mean() if early_mask.any() else np.nan
                )
        else:
            _set_nan(features, safe_name)

    return features

In [ ]:
# Quick test before full run
pos_stay  = cohort_clean[cohort_clean["respiratory_failure"] == 1].iloc[0]
neg_stay  = cohort_clean[cohort_clean["respiratory_failure"] == 0].iloc[0]

for label, stay in [("POSITIVE", pos_stay), ("NEGATIVE", neg_stay)]:
    stay_labs = labs_filtered[
        labs_filtered["hadm_id"] == int(stay["hadm_id"])
    ].copy()
    feats = extract_lab_features_v2(stay, stay_labs)
    print(f"{label}: window={stay['feature_window_hours']:.2f}h | "
          f"features={len(feats)} | "
          f"Creatinine_count={feats['Creatinine_count']}")

print(f"\nExpected features: {1 + len(TOP_LABS) * 8}")
print(f"Match: {len(feats) == 1 + len(TOP_LABS) * 8}")
print("Extractor functions ready ✓")

POSITIVE: window=0.38h | features=161 | Creatinine_count=0
NEGATIVE: window=12.00h | features=161 | Creatinine_count=1

Expected features: 161
Match: True
Extractor functions ready ✓


In [ ]:
# ── Cell 8: Checkpointed Lab Extraction v2 ───────────────────
checkpoint_dir_v2 = save_dir / "lab_checkpoints_v2"
checkpoint_dir_v2.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 500
n_stays    = len(cohort_clean)
n_batches  = (n_stays + BATCH_SIZE - 1) // BATCH_SIZE

In [ ]:
# Find already completed batches
completed_batches = set()
for f in checkpoint_dir_v2.iterdir():
    if f.name.startswith("batch_") and f.name.endswith(".csv"):
        batch_num = int(
            f.name.replace("batch_", "").replace(".csv", "")
        )
        completed_batches.add(batch_num)

print(f"Total stays:               {n_stays:,}")
print(f"Total batches:             {n_batches}")
print(f"Batches already completed: {len(completed_batches)}/{n_batches}")
if completed_batches:
    print(f"  Resuming from batch:   {max(completed_batches) + 1}")
else:
    print(f"  Starting fresh")

Total stays:               91,262
Total batches:             183
Batches already completed: 0/183
  Starting fresh


In [ ]:
start_time = time.time()

In [ ]:
for batch_idx in range(n_batches):

    if batch_idx in completed_batches:
        continue

    batch_start    = batch_idx * BATCH_SIZE
    batch_end      = min(batch_start + BATCH_SIZE, n_stays)
    batch_stays    = cohort_clean.iloc[batch_start:batch_end]
    batch_hadm_ids = set(batch_stays["hadm_id"].unique())

    batch_labs = labs_filtered[
        labs_filtered["hadm_id"].isin(batch_hadm_ids)
    ].copy()

    batch_features = []
    for _, stay_row in batch_stays.iterrows():
        stay_labs = batch_labs[
            batch_labs["hadm_id"] == stay_row["hadm_id"]
        ]
        feats = extract_lab_features_v2(stay_row, stay_labs)
        batch_features.append(feats)

    batch_df   = pd.DataFrame(batch_features)
    batch_file = checkpoint_dir_v2 / f"batch_{batch_idx:04d}.csv"
    batch_df.to_csv(batch_file, index=False)

    if (batch_idx + 1) % 10 == 0 or batch_idx == n_batches - 1:
        elapsed   = time.time() - start_time
        rate      = batch_end / elapsed if elapsed > 0 else 0
        remaining = (n_stays - batch_end) / rate if rate > 0 else 0
        print(f"  Batch {batch_idx+1:4d}/{n_batches} | "
              f"{batch_end:6,}/{n_stays:,} stays | "
              f"ETA: {remaining/60:.1f} min | "
              f"Saved ✓")

...  Batch   10/183 |  5,000/91,262 stays | ETA: 44.8 min | Saved ✓
..  Batch   20/183 | 10,000/91,262 stays | ETA: 41.6 min | Saved ✓
...  Batch   30/183 | 15,000/91,262 stays | ETA: 38.8 min | Saved ✓
..  Batch   40/183 | 20,000/91,262 stays | ETA: 36.3 min | Saved ✓
...  Batch   50/183 | 25,000/91,262 stays | ETA: 33.7 min | Saved ✓
..  Batch   60/183 | 30,000/91,262 stays | ETA: 31.0 min | Saved ✓
...  Batch   70/183 | 35,000/91,262 stays | ETA: 28.6 min | Saved ✓
..  Batch   80/183 | 40,000/91,262 stays | ETA: 26.2 min | Saved ✓
...  Batch   90/183 | 45,000/91,262 stays | ETA: 23.6 min | Saved ✓
..  Batch  100/183 | 50,000/91,262 stays | ETA: 21.0 min | Saved ✓
...  Batch  110/183 | 55,000/91,262 stays | ETA: 18.4 min | Saved ✓
...  Batch  120/183 | 60,000/91,262 stays | ETA: 15.9 min | Saved ✓
..  Batch  130/183 | 65,000/91,262 stays | ETA: 13.4 min | Saved ✓
...  Batch  140/183 | 70,000/91,262 stays | ETA: 10.8 min | Saved ✓
..  Batch  150/183 | 75,000/91,262 stays | ETA: 8.3 mi

In [ ]:
# Combine all batches
print("\nCombining all batch files...")
all_batch_files = sorted(checkpoint_dir_v2.glob("batch_*.csv"))
lab_features_v2 = pd.concat(
    [pd.read_csv(f) for f in all_batch_files],
    ignore_index=True
)


Combining all batch files...


In [ ]:
print(f"\nFinal shape:  {lab_features_v2.shape}")
print(f"Rows match:   {len(lab_features_v2) == n_stays}")
print(f"Cols match:   {lab_features_v2.shape[1] == 161}")


Final shape:  (91262, 161)
Rows match:   True
Cols match:   True


In [ ]:
lab_features_v2.to_csv(save_dir / "lab_features_v2.csv", index=False)
print(f"✓ lab_features_v2.csv saved to Drive")

.✓ lab_features_v2.csv saved to Drive


In [ ]:
missing_pct = lab_features_v2.isnull().mean() * 100
print(f"\nMissing rate summary:")
print(f"  Mean:  {missing_pct.mean():.1f}%")
print(f"  Max:   {missing_pct.max():.1f}%")
print(f"  Min:   {missing_pct.min():.1f}%")
print("\nLab extraction complete ✓")


Missing rate summary:
  Mean:  26.8%
  Max:   55.3%
  Min:   0.0%

Lab extraction complete ✓


In [ ]:
#PRESCRIPTION FEATURE EXTRACTION

In [ ]:
# ── Cell 9: Prescription Feature Extraction ───────────────────

# Filter prescriptions to clean cohort
presc_filtered = prescriptions[
    prescriptions["hadm_id"].isin(
        set(cohort_clean["hadm_id"].unique())
    )
].copy()
presc_filtered["hadm_id"] = presc_filtered["hadm_id"].astype("int64")
print(f"Prescriptions for clean cohort: {len(presc_filtered):,}")

Prescriptions for clean cohort: 8,482,752


In [ ]:
# ── Drug category definitions ─────────────────────────────────
# These are clinically meaningful groups
# NOTE: vasopressors are NOT included — they are correlated
# with the outcome but are not part of the label definition
# so they are safe, but we exclude them to keep features
# as early-warning signals only

DRUG_CATEGORIES = {
    "antibiotic": [
        "vancomycin", "piperacillin", "cefepime", "meropenem",
        "ciprofloxacin", "metronidazole", "levofloxacin",
        "ampicillin", "ceftriaxone", "azithromycin"
    ],
    "sedation": [
        "propofol", "midazolam", "lorazepam", "dexmedetomidine",
        "ketamine", "diazepam"
    ],
    "opioid": [
        "fentanyl", "morphine", "hydromorphone", "dilaudid",
        "oxycodone", "methadone", "remifentanil"
    ],
    "cardiac": [
        "metoprolol", "amiodarone", "digoxin", "diltiazem",
        "lisinopril", "carvedilol", "atenolol", "labetalol"
    ],
    "anticoagulant": [
        "heparin", "warfarin", "enoxaparin", "apixaban",
        "rivaroxaban", "fondaparinux"
    ],
    "insulin": [
        "insulin"
    ],
    "diuretic": [
        "furosemide", "torsemide", "bumetanide", "spironolactone"
    ],
    "steroid": [
        "hydrocortisone", "methylprednisolone", "dexamethasone",
        "prednisone", "fludrocortisone"
    ]
}

In [ ]:
# IV routes — continuous drips are most serious
IV_ROUTES     = {"iv", "iv drip", "iv bolus"}
SERIOUS_ROUTE = "iv drip"

In [ ]:
def extract_presc_features(stay_row, stay_presc):
    """
    Extract prescription features for a single ICU stay.
    Uses feature_cutoff to prevent leakage.

    Features:
      - total_presc        : total orders before cutoff
      - unique_drugs       : number of distinct drugs
      - iv_count           : IV route orders
      - iv_drip_count      : continuous IV drip orders (severity)
      - iv_drip_ratio      : iv_drip_count / total (0 if no presc)
      - has_{category}     : binary flag per drug category (8 cats)

    Total: 5 count/ratio + 8 binary = 13 features + stay_id
    """
    features   = {"stay_id": stay_row["stay_id"]}
    icu_intime = stay_row["icu_intime"]
    cutoff     = stay_row["feature_cutoff"]

    if len(stay_presc) > 0:
        stay_presc = stay_presc.copy()

        # Filter to window: admission up to cutoff
        # Prescriptions have no pre-admission window
        # (unlike labs which can be drawn in ED before ICU)
        stay_presc = stay_presc[
            (stay_presc["med_starttime"] >= icu_intime) &
            (stay_presc["med_starttime"] <= cutoff)
        ].copy()

    n_total = len(stay_presc)

    # Count features
    features["total_presc"]  = n_total
    features["unique_drugs"] = (
        stay_presc["drug"].nunique() if n_total > 0 else 0
    )

    # Route-based features
    if n_total > 0:
        routes_lower = stay_presc["route"].astype(str).str.lower()
        iv_mask      = routes_lower.isin(IV_ROUTES)
        drip_mask    = routes_lower == SERIOUS_ROUTE

        features["iv_count"]      = int(iv_mask.sum())
        features["iv_drip_count"] = int(drip_mask.sum())
        features["iv_drip_ratio"] = features["iv_drip_count"] / n_total
    else:
        features["iv_count"]      = 0
        features["iv_drip_count"] = 0
        features["iv_drip_ratio"] = 0.0

    # Drug category flags
    if n_total > 0:
        drugs_lower = stay_presc["drug"].astype(str).str.lower()
        for category, keywords in DRUG_CATEGORIES.items():
            features[f"has_{category}"] = int(
                drugs_lower.apply(
                    lambda d: any(kw in d for kw in keywords)
                ).any()
            )
    else:
        for category in DRUG_CATEGORIES:
            features[f"has_{category}"] = 0

    return features

In [ ]:
# ── Test on one positive and one negative case ─────────────────
pos_stay = cohort_clean[cohort_clean["respiratory_failure"] == 1].iloc[0]
neg_stay = cohort_clean[cohort_clean["respiratory_failure"] == 0].iloc[0]

for label, stay in [("POSITIVE", pos_stay), ("NEGATIVE", neg_stay)]:
    stay_presc = presc_filtered[
        presc_filtered["hadm_id"] == int(stay["hadm_id"])
    ].copy

In [ ]:
for label, stay in [("POSITIVE", pos_stay), ("NEGATIVE", neg_stay)]:
    stay_presc = presc_filtered[
        presc_filtered["hadm_id"] == int(stay["hadm_id"])
    ].copy()
    feats = extract_presc_features(stay, stay_presc)
    print(f"\n{label} case (stay_id {stay['stay_id']}):")
    print(f"  feature_window:  {stay['feature_window_hours']:.2f}h")
    print(f"  total_presc:     {feats['total_presc']}")
    print(f"  unique_drugs:    {feats['unique_drugs']}")
    print(f"  iv_count:        {feats['iv_count']}")
    print(f"  iv_drip_count:   {feats['iv_drip_count']}")
    print(f"  iv_drip_ratio:   {feats['iv_drip_ratio']:.3f}")
    print(f"  has_antibiotic:  {feats['has_antibiotic']}")
    print(f"  has_sedation:    {feats['has_sedation']}")
    print(f"  has_opioid:      {feats['has_opioid']}")
    print(f"  Feature count:   {len(feats)}")

expected = 1 + 5 + len(DRUG_CATEGORIES)
print(f"\nExpected features: {expected}")
print(f"Match: {len(feats) == expected}")


POSITIVE case (stay_id 37081114):
  feature_window:  0.38h
  total_presc:     2
  unique_drugs:    2
  iv_count:        0
  iv_drip_count:   0
  iv_drip_ratio:   0.000
  has_antibiotic:  0
  has_sedation:    0
  has_opioid:      0
  Feature count:   14

NEGATIVE case (stay_id 39553978):
  feature_window:  12.00h
  total_presc:     18
  unique_drugs:    16
  iv_count:        2
  iv_drip_count:   0
  iv_drip_ratio:   0.000
  has_antibiotic:  0
  has_sedation:    0
  has_opioid:      0
  Feature count:   14

Expected features: 14
Match: True


In [ ]:
# ── Cell 10: Checkpointed Prescription Extraction ─────────────

checkpoint_dir_presc = save_dir / "presc_checkpoints"
checkpoint_dir_presc.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 500
n_stays    = len(cohort_clean)
n_batches  = (n_stays + BATCH_SIZE - 1) // BATCH_SIZE

# Find already completed batches
completed_batches = set()
for f in checkpoint_dir_presc.iterdir():
    if f.name.startswith("batch_") and f.name.endswith(".csv"):
        batch_num = int(
            f.name.replace("batch_", "").replace(".csv", "")
        )
        completed_batches.add(batch_num)

print(f"Total stays:               {n_stays:,}")
print(f"Total batches:             {n_batches}")
print(f"Batches already completed: {len(completed_batches)}/{n_batches}")
if completed_batches:
    print(f"  Resuming from batch:   {max(completed_batches) + 1}")
else:
    print(f"  Starting fresh")

start_time = time.time()

for batch_idx in range(n_batches):

    if batch_idx in completed_batches:
        continue

    batch_start    = batch_idx * BATCH_SIZE
    batch_end      = min(batch_start + BATCH_SIZE, n_stays)
    batch_stays    = cohort_clean.iloc[batch_start:batch_end]
    batch_hadm_ids = set(batch_stays["hadm_id"].unique())

    batch_presc = presc_filtered[
        presc_filtered["hadm_id"].isin(batch_hadm_ids)
    ].copy()

    batch_features = []
    for _, stay_row in batch_stays.iterrows():
        stay_presc = batch_presc[
            batch_presc["hadm_id"] == stay_row["hadm_id"]
        ]
        feats = extract_presc_features(stay_row, stay_presc)
        batch_features.append(feats)

    batch_df   = pd.DataFrame(batch_features)
    batch_file = checkpoint_dir_presc / f"batch_{batch_idx:04d}.csv"
    batch_df.to_csv(batch_file, index=False)

    if (batch_idx + 1) % 10 == 0 or batch_idx == n_batches - 1:
        elapsed   = time.time() - start_time
        rate      = batch_end / elapsed if elapsed > 0 else 0
        remaining = (n_stays - batch_end) / rate if rate > 0 else 0
        print(f"  Batch {batch_idx+1:4d}/{n_batches} | "
              f"{batch_end:6,}/{n_stays:,} stays | "
              f"ETA: {remaining/60:.1f} min | "
              f"Saved ✓")

# Combine all batches
print("\nCombining all batch files...")
all_batch_files  = sorted(checkpoint_dir_presc.glob("batch_*.csv"))
presc_features   = pd.concat(
    [pd.read_csv(f) for f in all_batch_files],
    ignore_index=True
)

print(f"\nFinal shape:  {presc_features.shape}")
print(f"Rows match:   {len(presc_features) == n_stays}")
print(f"Cols match:   {presc_features.shape[1] == 14}")

presc_features.to_csv(save_dir / "presc_features.csv", index=False)
print(f"✓ presc_features.csv saved to Drive")

# Missing rate check
missing_pct = presc_features.isnull().mean() * 100
print(f"\nMissing rate summary:")
print(f"  Mean:  {missing_pct.mean():.1f}%")
print(f"  Max:   {missing_pct.max():.1f}%")
print(f"  Min:   {missing_pct.min():.1f}%")

# Distribution of key features
print(f"\nKey feature distributions:")
print(f"  total_presc mean:     {presc_features['total_presc'].mean():.1f}")
print(f"  unique_drugs mean:    {presc_features['unique_drugs'].mean():.1f}")
print(f"  iv_drip_count mean:   {presc_features['iv_drip_count'].mean():.1f}")
for cat in DRUG_CATEGORIES:
    pct = presc_features[f"has_{cat}"].mean() * 100
    print(f"  has_{cat:<15} {pct:.1f}% of stays")

print("\nPrescription extraction complete ✓")

Total stays:               91,262
Total batches:             183
Batches already completed: 0/183
  Starting fresh
.  Batch   10/183 |  5,000/91,262 stays | ETA: 3.2 min | Saved ✓
  Batch   20/183 | 10,000/91,262 stays | ETA: 3.0 min | Saved ✓
  Batch   30/183 | 15,000/91,262 stays | ETA: 2.9 min | Saved ✓
  Batch   40/183 | 20,000/91,262 stays | ETA: 2.7 min | Saved ✓
  Batch   50/183 | 25,000/91,262 stays | ETA: 2.6 min | Saved ✓
.  Batch   60/183 | 30,000/91,262 stays | ETA: 2.4 min | Saved ✓
  Batch   70/183 | 35,000/91,262 stays | ETA: 2.2 min | Saved ✓
  Batch   80/183 | 40,000/91,262 stays | ETA: 2.0 min | Saved ✓
  Batch   90/183 | 45,000/91,262 stays | ETA: 1.8 min | Saved ✓
  Batch  100/183 | 50,000/91,262 stays | ETA: 1.6 min | Saved ✓
.  Batch  110/183 | 55,000/91,262 stays | ETA: 1.4 min | Saved ✓
  Batch  120/183 | 60,000/91,262 stays | ETA: 1.2 min | Saved ✓
  Batch  130/183 | 65,000/91,262 stays | ETA: 1.0 min | Saved ✓
  Batch  140/183 | 70,000/91,262 stays | ETA: 0.8 

In [ ]:
# ── Cell 11: Assemble Final Feature Table ─────────────────────

# Administrative features from cohort_clean
admin_features = cohort_clean[[
    "stay_id",
    "anchor_age",
    "sex",
    "stay_duration_hours",
    "obs_window_hours",
    "is_short_stay",
    "feature_window_hours",
    "respiratory_failure"   # label
]].copy()

# Encode sex as binary
admin_features["sex"] = (
    admin_features["sex"] == "M"
).astype(int)

print(f"Admin features shape:  {admin_features.shape}")
print(f"Lab features shape:    {lab_features_v2.shape}")
print(f"Presc features shape:  {presc_features.shape}")

# Merge all three on stay_id
final_df = (
    admin_features
    .merge(lab_features_v2,  on="stay_id", how="left")
    .merge(presc_features,   on="stay_id", how="left")
)

print(f"\nFinal merged shape:    {final_df.shape}")
print(f"Rows match:            {len(final_df) == len(cohort_clean)}")
print(f"stay_id unique:        {final_df['stay_id'].nunique() == len(final_df)}")

# Expected column count
# 8 admin (incl label) + 160 lab + 13 presc = 181 + stay_id = 182
expected_cols = 1 + 7 + 160 + 13
print(f"\nExpected columns:      {expected_cols}")
print(f"Actual columns:        {final_df.shape[1]}")

# Label distribution check
print(f"\nLabel distribution:")
print(final_df["respiratory_failure"].value_counts())
print(f"Positive rate: {final_df['respiratory_failure'].mean()*100:.1f}%")

# Overall missing rate
missing_pct = final_df.isnull().mean() * 100
print(f"\nOverall missing rate:")
print(f"  Mean:  {missing_pct.mean():.1f}%")
print(f"  Max:   {missing_pct.max():.1f}%")
print(f"  Min:   {missing_pct.min():.1f}%")

# Missing by feature group
lab_cols   = [c for c in final_df.columns
              if any(c.endswith(s) for s in
                     ["_mean","_min","_max","_count",
                      "_first","_last","_delta","_early_mean"])
              and c != "stay_id"]
presc_cols = [c for c in final_df.columns
              if c.startswith("has_") or c in
              ["total_presc","unique_drugs",
               "iv_count","iv_drip_count","iv_drip_ratio"]]
admin_cols = ["anchor_age","sex","stay_duration_hours",
              "obs_window_hours","is_short_stay","feature_window_hours"]

print(f"\nMissing rate by group:")
print(f"  Admin features:  {missing_pct[admin_cols].mean():.1f}%")
print(f"  Lab features:    {missing_pct[lab_cols].mean():.1f}%")
print(f"  Presc features:  {missing_pct[presc_cols].mean():.1f}%")

# Save final table
final_df.to_csv(save_dir / "final_features.csv", index=False)
print(f"\n✓ final_features.csv saved to Drive")
print(f"  Shape: {final_df.shape}")

Admin features shape:  (91262, 8)
Lab features shape:    (91262, 161)
Presc features shape:  (91262, 14)

Final merged shape:    (91262, 181)
Rows match:            True
stay_id unique:        True

Expected columns:      181
Actual columns:        181

Label distribution:
respiratory_failure
0    56434
1    34828
Name: count, dtype: int64
Positive rate: 38.2%

Overall missing rate:
  Mean:  23.8%
  Max:   55.3%
  Min:   0.0%

Missing rate by group:
  Admin features:  0.0%
  Lab features:    26.6%
  Presc features:  0.0%

✓ final_features.csv saved to Drive
  Shape: (91262, 181)


In [ ]:
'''
WHAT WE'VE BUILT SO FAR
final_features.csv — 91,262 stays × 181 columns

Column breakdown:
  1   stay_id
  7   admin features  (age, sex, stay duration, window info, label)
  160 lab features    (20 labs × 8 statistics each)
  13  presc features  (counts, ratios, 8 drug category flags)

Label: 38.2% positive (respiratory failure)
       61.8% negative (stable)

Missing rates:
  Admin:  0.0%  ← perfect
  Presc:  0.0%  ← perfect
  Labs:   26.6% ← expected, handled by imputation later
  '''

"\nWHAT WE'VE BUILT SO FAR\nfinal_features.csv — 91,262 stays × 181 columns\n\nColumn breakdown:\n  1   stay_id\n  7   admin features  (age, sex, stay duration, window info, label)\n  160 lab features    (20 labs × 8 statistics each)\n  13  presc features  (counts, ratios, 8 drug category flags)\n\nLabel: 38.2% positive (respiratory failure)\n       61.8% negative (stable)\n\nMissing rates:\n  Admin:  0.0%  ← perfect\n  Presc:  0.0%  ← perfect\n  Labs:   26.6% ← expected, handled by imputation later\n  "

In [ ]:
# ── Cell 12: Clinical Text Serialization ──────────────────────

def format_lab_value(value, decimals=1):
    """Format a lab value cleanly, return None if missing."""
    if pd.isna(value):
        return None
    return f"{value:.{decimals}f}"


def row_to_clinical_text(row):
    """
    Convert a patient's structured features into a clinical
    text summary for ClinicalBERT.

    Writes in the style of a clinical handoff note —
    the kind of text ClinicalBERT was pretrained on.
    """
    parts = []

    # ── Demographics ──────────────────────────────────────────
    age     = int(row["anchor_age"])
    sex_str = "male" if row["sex"] == 1 else "female"
    parts.append(f"Patient is a {age}-year-old {sex_str}.")

    # ── Observation window ────────────────────────────────────
    window = row["feature_window_hours"]
    if window < 1:
        parts.append(
            f"Clinical data available for {window*60:.0f} minutes "
            f"following ICU admission."
        )
    else:
        parts.append(
            f"Clinical data available for {window:.1f} hours "
            f"following ICU admission."
        )

    # ── Lab values — only mention labs that have data ─────────
    # Group by organ system for clinical readability
    lab_groups = {
        "Metabolic panel": [
            ("Glucose",        "Glucose",       "mg/dL"),
            ("Sodium",         "Sodium",        "mEq/L"),
            ("Potassium",      "Potassium",     "mEq/L"),
            ("Chloride",       "Chloride",      "mEq/L"),
            ("Bicarbonate",    "Bicarbonate",   "mEq/L"),
            ("Anion_Gap",      "Anion Gap",     "mEq/L"),
            ("Calcium_Total",  "Calcium",       "mg/dL"),
            ("Magnesium",      "Magnesium",     "mg/dL"),
            ("Phosphate",      "Phosphate",     "mg/dL"),
        ],
        "Renal function": [
            ("Creatinine",     "Creatinine",    "mg/dL"),
            ("Urea_Nitrogen",  "BUN",           "mg/dL"),
        ],
        "Blood count": [
            ("Hemoglobin",        "Hemoglobin",      "g/dL"),
            ("Hematocrit",        "Hematocrit",      "%"),
            ("White_Blood_Cells", "WBC",             "K/uL"),
            ("Platelet_Count",    "Platelets",       "K/uL"),
            ("Red_Blood_Cells",   "RBC",             "M/uL"),
            ("MCHC",              "MCHC",            "g/dL"),
            ("MCV",               "MCV",             "fL"),
            ("MCH",               "MCH",             "pg"),
            ("RDW",               "RDW",             "%"),
        ]
    }

    for group_name, labs in lab_groups.items():
        lab_mentions = []
        for safe_name, display_name, unit in labs:
            mean_val  = row.get(f"{safe_name}_mean")
            delta_val = row.get(f"{safe_name}_delta")
            count_val = row.get(f"{safe_name}_count", 0)

            if pd.isna(mean_val) or count_val == 0:
                continue

            mention = f"{display_name} {mean_val:.1f} {unit}"

            # Add trajectory if meaningful and more than one reading
            if not pd.isna(delta_val) and count_val > 1:
                if delta_val > 0:
                    mention += f" (rising)"
                elif delta_val < 0:
                    mention += f" (falling)"

            lab_mentions.append(mention)

        if lab_mentions:
            parts.append(
                f"{group_name}: {', '.join(lab_mentions)}."
            )

    # ── Prescription summary ──────────────────────────────────
    n_presc    = int(row["total_presc"])
    n_unique   = int(row["unique_drugs"])
    n_iv_drip  = int(row["iv_drip_count"])

    if n_presc == 0:
        parts.append("No medications ordered during observation window.")
    else:
        parts.append(
            f"Received {n_presc} medication orders "
            f"({n_unique} unique drugs) during observation window."
        )
        if n_iv_drip > 0:
            parts.append(
                f"{n_iv_drip} continuous IV "
                f"{'drip' if n_iv_drip == 1 else 'drips'} running."
            )

    # ── Drug categories present ───────────────────────────────
    active_categories = []
    category_display  = {
        "antibiotic":    "antibiotics",
        "sedation":      "sedatives",
        "opioid":        "opioids",
        "cardiac":       "cardiac medications",
        "anticoagulant": "anticoagulation",
        "insulin":       "insulin",
        "diuretic":      "diuretics",
        "steroid":       "corticosteroids"
    }
    for cat, display in category_display.items():
        if row.get(f"has_{cat}", 0) == 1:
            active_categories.append(display)

    if active_categories:
        parts.append(
            f"Active medications include: "
            f"{', '.join(active_categories)}."
        )

    return " ".join(parts)


# ── Apply to all patients ──────────────────────────────────────
print("Generating clinical text for all patients...")
start = time.time()

final_df["clinical_text"] = final_df.apply(
    row_to_clinical_text, axis=1
)

elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s")

# ── Show examples ──────────────────────────────────────────────
print("\n" + "="*60)
print("POSITIVE CASE EXAMPLE:")
print("="*60)
pos_example = final_df[
    final_df["respiratory_failure"] == 1
].iloc[0]
print(pos_example["clinical_text"])
print(f"\nLabel: {pos_example['respiratory_failure']}")
print(f"Feature window: {pos_example['feature_window_hours']:.2f}h")

print("\n" + "="*60)
print("NEGATIVE CASE EXAMPLE:")
print("="*60)
neg_example = final_df[
    final_df["respiratory_failure"] == 0
].iloc[0]
print(neg_example["clinical_text"])
print(f"\nLabel: {neg_example['respiratory_failure']}")
print(f"Feature window: {neg_example['feature_window_hours']:.2f}h")

# ── Text length statistics ─────────────────────────────────────
text_lengths = final_df["clinical_text"].str.split().str.len()
print(f"\nClinical text length (words):")
print(f"  Mean:   {text_lengths.mean():.0f}")
print(f"  Median: {text_lengths.median():.0f}")
print(f"  Min:    {text_lengths.min()}")
print(f"  Max:    {text_lengths.max()}")
print(f"  Under 512 tokens: almost certainly all"
      f" (BERT max is 512)")

# ── Save with text included ────────────────────────────────────
final_df.to_csv(save_dir / "final_features_with_text.csv", index=False)
print(f"\n✓ final_features_with_text.csv saved to Drive")
print(f"  Shape: {final_df.shape}")

Generating clinical text for all patients...
Done in 14.9s

POSITIVE CASE EXAMPLE:
Patient is a 86-year-old female. Clinical data available for 23 minutes following ICU admission. Received 2 medication orders (2 unique drugs) during observation window. Active medications include: anticoagulation.

Label: 1
Feature window: 0.38h

NEGATIVE CASE EXAMPLE:
Patient is a 52-year-old female. Clinical data available for 12.0 hours following ICU admission. Metabolic panel: Glucose 115.0 mg/dL, Sodium 132.0 mEq/L, Potassium 4.7 mEq/L, Chloride 102.0 mEq/L, Bicarbonate 21.0 mEq/L, Anion Gap 14.0 mEq/L, Calcium 9.3 mg/dL, Magnesium 2.3 mg/dL, Phosphate 2.4 mg/dL. Renal function: Creatinine 0.5 mg/dL, BUN 33.0 mg/dL. Received 18 medication orders (16 unique drugs) during observation window. Active medications include: anticoagulation.

Label: 0
Feature window: 12.00h

Clinical text length (words):
  Mean:   84
  Median: 97
  Min:    20
  Max:    128
  Under 512 tokens: almost certainly all (BERT max

In [ ]:
# ── Cell 13: Patient-Level Train/Val/Test Split (70/15/15) ────
from sklearn.model_selection import GroupShuffleSplit

# Get subject_id for each stay
stay_to_subject = cohort_clean[["stay_id", "subject_id"]].set_index("stay_id")["subject_id"]
final_df["subject_id"] = final_df["stay_id"].map(stay_to_subject)

print(f"Unique patients: {final_df['subject_id'].nunique():,}")
print(f"Total stays:     {len(final_df):,}")

# Split 1: 70% train, 30% temp
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss1.split(final_df, groups=final_df["subject_id"]))
train_df = final_df.iloc[train_idx].copy().reset_index(drop=True)
temp_df  = final_df.iloc[temp_idx].copy().reset_index(drop=True)

# Split 2: 50/50 of temp → val (15%) and test (15%)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["subject_id"]))
val_df  = temp_df.iloc[val_idx].copy().reset_index(drop=True)
test_df = temp_df.iloc[test_idx].copy().reset_index(drop=True)

# ── Verify zero patient overlap ────────────────────────────────
assert len(set(train_df.subject_id) & set(val_df.subject_id))  == 0, "Train/val leak!"
assert len(set(train_df.subject_id) & set(test_df.subject_id)) == 0, "Train/test leak!"
assert len(set(val_df.subject_id)   & set(test_df.subject_id)) == 0, "Val/test leak!"

print(f"\nTrain : {len(train_df):,} stays | {train_df.subject_id.nunique():,} patients | "
      f"RF rate {train_df['respiratory_failure'].mean()*100:.1f}%")
print(f"Val   : {len(val_df):,} stays | {val_df.subject_id.nunique():,} patients | "
      f"RF rate {val_df['respiratory_failure'].mean()*100:.1f}%")
print(f"Test  : {len(test_df):,} stays | {test_df.subject_id.nunique():,} patients | "
      f"RF rate {test_df['respiratory_failure'].mean()*100:.1f}%")
print(f"\nPatient overlap: train/val={len(set(train_df.subject_id)&set(val_df.subject_id))} "
      f"train/test={len(set(train_df.subject_id)&set(test_df.subject_id))} "
      f"val/test={len(set(val_df.subject_id)&set(test_df.subject_id))}")
print("✓ Clean three-way split — zero overlap")

# ── Save ───────────────────────────────────────────────────────
train_df.to_csv(save_dir / "train_df.csv", index=False)
val_df.to_csv(  save_dir / "val_df.csv",   index=False)
test_df.to_csv( save_dir / "test_df.csv",  index=False)
print("\n✓ train_df.csv saved")
print("✓ val_df.csv saved")
print("✓ test_df.csv saved")

Unique patients: 65,366
Total stays:     91,262

Train set: 72,971 stays
Test set:  18,291 stays
Split ratio: 20.0% test

Patient overlap between train/test: 0
  ✓ Clean split

Label distribution:
  Train positive rate: 38.1%
  Test positive rate:  38.3%

Unique patients:
  Train: 52,292
  Test:  13,074

✓ train_df.csv saved
✓ test_df.csv saved


In [ ]:
'''
CLINICALBERT FINE-TUNING
'''

'\nCLINICALBERT FINE-TUNING\n'

In [ ]:
save_dir = Path("/content/drive/MyDrive/USC/ICU-MM-main/outputs")

train_df = pd.read_csv(save_dir / "train_df.csv")
test_df  = pd.read_csv(save_dir / "test_df.csv")

print(f"train_df shape: {train_df.shape}")
print(f"test_df shape:  {test_df.shape}")
print(f"Train positive rate: {train_df['respiratory_failure'].mean()*100:.1f}%")
print(f"Test positive rate:  {test_df['respiratory_failure'].mean()*100:.1f}%")
print(f"clinical_text present: {'clinical_text' in train_df.columns}")
print("Splits loaded ✓")

train_df shape: (72971, 183)
test_df shape:  (18291, 183)
Train positive rate: 38.1%
Test positive rate:  38.3%
clinical_text present: True
Splits loaded ✓


In [ ]:
# ── Cell 14 (NEW): Fix multimodal_df — ensure subject_id + stay_id present ───
multimodal_df = pd.read_csv(save_dir / "multimodal_df.csv")

# Verify stay_id is present
assert "stay_id" in multimodal_df.columns, "stay_id missing from multimodal_df!"

# Add subject_id if missing
if "subject_id" not in multimodal_df.columns:
    stay_to_subject = cohort_clean.set_index("stay_id")["subject_id"]
    multimodal_df.insert(0, "subject_id", multimodal_df["stay_id"].map(stay_to_subject))
    print("subject_id added ✓")
else:
    print("subject_id already present ✓")

# Reorder so identifiers are the first two columns
id_cols    = ["subject_id", "stay_id"]
other_cols = [c for c in multimodal_df.columns if c not in id_cols]
multimodal_df = multimodal_df[id_cols + other_cols]

# Verify
assert multimodal_df.columns[0] == "subject_id", "subject_id must be column 0"
assert multimodal_df.columns[1] == "stay_id",    "stay_id must be column 1"
assert multimodal_df["subject_id"].isna().sum() == 0, "subject_id has NaN rows!"

print(f"Shape:             {multimodal_df.shape}")
print(f"First two columns: {list(multimodal_df.columns[:2])}")
print(f"subject_id unique: {multimodal_df['subject_id'].nunique()}")
print(f"stay_id unique:    {multimodal_df['stay_id'].nunique()}")

# Overwrite with corrected version
multimodal_df.to_csv(save_dir / "multimodal_df.csv", index=False)
print("✓ multimodal_df.csv overwritten with subject_id as first column")